# Homework 1: Setup and First Captures
## Name: Jon Knapp
## Date: October 6, 2026

### Smoke Test

In [7]:
# A tiny pass/fail collector used by the rest of the notebook.
import sys, platform, textwrap

RESULTS = []

def check(name, ok, detail="", fix=""):
    RESULTS.append((name, bool(ok), detail, fix))
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f"  {detail}" if detail else ""))
    if not ok and fix:
        print(textwrap.indent(textwrap.fill(f"Fix: {fix}", 86), "       "))
    return ok

print("Python  ", sys.version.split()[0])
print("Platform", platform.platform())
print("Executable", sys.executable)

Python   3.13.7
Platform Windows-11-10.0.26300-SP0
Executable C:\Users\jknap\AppData\Local\Programs\Python\Python313\python.exe


In [8]:
# 1. Python version. The course targets 3.12; 3.10 or newer is fine.
major, minor = sys.version_info[:2]
check(
    "Python 3.10 or newer",
    (major, minor) >= (3, 10),
    f"found {major}.{minor}",
    "You should not see this if you started the notebook with `uv run jupyter lab` "
    "from the course folder. Check that pyproject.toml and uv.lock are in that "
    "folder, then run `uv sync` again.",
)

[PASS] Python 3.10 or newer  found 3.13


True

In [9]:
import importlib

REQUIRED = ["numpy", "scipy", "matplotlib", "cv2", "rawpy", "imageio", "skimage"]
LATER    = ["torch", "colour", "gradio"]

def version_of(mod):
    for attr in ("__version__", "version", "VERSION"):
        v = getattr(mod, attr, None)
        if isinstance(v, str):
            return v
    return "?"

def try_import(names, required):
    found = {}
    for name in names:
        try:
            mod = importlib.import_module(name)
            found[name] = version_of(mod)
        except Exception as exc:
            found[name] = None
            if required:
                check(f"import {name}", False, "",
                      f"{PIP_NAME.get(name, name)} is missing. Run `uv sync` in the course "
                      f"folder; do not pip install it.   ({type(exc).__name__})")
    return found

# Distribution names, for the message only. Do not pip install these by hand: the
# environment comes from uv.lock, and installing plain opencv-python on top of
# opencv-contrib-python breaks cv2 in a way that is hard to diagnose.
PIP_NAME = {"cv2": "opencv-contrib-python", "skimage": "scikit-image",
            "colour": "colour-science"}

req = try_import(REQUIRED, required=True)
check("all required packages import", all(v is not None for v in req.values()),
      ", ".join(f"{k} {v}" for k, v in req.items() if v))

later = try_import(LATER, required=False)
missing_later = [k for k, v in later.items() if v is None]
print()
print("Not used until week 6, but the course environment installs them now:",
      ", ".join(f"{k} {v}" for k, v in later.items() if v) or "(none installed)")
if missing_later:
    print("  missing:", ", ".join(missing_later), "- run `uv sync` again.")

[FAIL] import rawpy
       Fix: rawpy is missing. Run `uv sync` in the course folder; do not pip install it.
       (ModuleNotFoundError)
[FAIL] import imageio
       Fix: imageio is missing. Run `uv sync` in the course folder; do not pip install it.
       (ModuleNotFoundError)
[FAIL] import skimage
       Fix: scikit-image is missing. Run `uv sync` in the course folder; do not pip install
       it.   (ModuleNotFoundError)
[FAIL] all required packages import  numpy 2.4.4, scipy 1.17.1, matplotlib 3.10.8, cv2 4.13.0

Not used until week 6, but the course environment installs them now: torch 2.11.0+cpu
  missing: colour, gradio - run `uv sync` again.
